In [0]:
from pyspark.sql.functions import*
from pyspark.sql.types import*
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [0]:
EH_NAMESPACE=""
EH_NAME=""
EH_CONN_STR=""

In [0]:
KAFKA_OPTIONS = {
  "kafka.bootstrap.servers"  : f"{EH_NAMESPACE}.servicebus.windows.net:9093",
  "subscribe"                : EH_NAME,
  "kafka.sasl.mechanism"     : "PLAIN",
  "kafka.security.protocol"  : "SASL_SSL",
  "kafka.sasl.jaas.config"   : f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username=\"$ConnectionString\" password=\"{EH_CONN_STR}\";",
  "kafka.request.timeout.ms" : "60000",
  "kafka.session.timeout.ms" : "30000",
  "maxOffsetsPerTrigger"     :"50000",
  "failOnDataLoss"           : "true",
  "startingOffsets"          : "earliest",
}


In [0]:
df_raw = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

display(df_raw, checkpointLocation="/Volumes/restaurantopcat/00landing/checkpoints/eventhub_raw")

In [0]:
for q in spark.streams.active:
    q.stop()

In [0]:
spark.sql("DROP TABLE IF EXISTS `restaurantopcat`.`00landing`.`orders_parsed`")

In [0]:
orders_schema = StructType([ StructField("order_id",
 StringType(), True), StructField("timestamp", StringType(), True), StructField("restaurant_id", StringType(), True), StructField("customer_id", StringType(), True), StructField("order_type", StringType(), True), StructField("items", StringType(), True), StructField("total_amount", DoubleType(), True), StructField("payment_method", StringType(), True), StructField("order_status", StringType(), True) ])

In [0]:
from pyspark.sql.functions import from_json, col

df_parsed = (
    df_raw
    .withColumn("key_str", col("key").cast("string"))
    .withColumn("value_str", col("value").cast("string"))
    .withColumn("data", from_json(col("value_str"), orders_schema))
    .select("data.*")
    .withColumnRenamed("timestamp", "order_timestamp")
)

query = (
    df_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/restaurantopcat/00landing/checkpoints/orders_parsed_v2")
    .trigger(availableNow=True)
    .toTable("`restaurantopcat`.`00landing`.`orders_parsed`")
)
query.awaitTermination()

In [0]:
display(spark.table("`restaurantopcat`.`00landing`.`eventhub_raw`"))

In [0]:
display(spark.table("`restaurantopcat`.`00landing`.`orders_parsed`"))